# 主线替代路线对比

- 对应论文章节：第3.3.1节 主线替代路线
- 源脚本：`experiments/03_证伪实验/scripts/运行_主线替代路线对比.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 统一展示双片段证据保留、证据标注、局部窗口改写、模板引导、支持表综合等主线附近替代路线。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/03_证伪实验/scripts/运行_主线替代路线对比.py --suite all
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""运行证伪实验中的主线替代路线对比。

这个脚本覆盖论文 3.3.1 中三类最直接的替代路线：
1. 双片段证据保留；
2. 候选证据重写与标注；
3. 结构化综合路线。
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

RESULT_ROOT = ROOT / "results" / "01_主线替代路线"
CRUD_SUBSET_SPLITS = {"questanswer_2docs": 797, "questanswer_3docs": 797}

SUITES: dict[str, dict[str, Any]] = {
    "dual_snippet": {
        "output_root": RESULT_ROOT / "双片段路线_20260517",
        "variants": (
            PipelineVariant(
                key="baseline_rrf_rerank_direct",
                label="基线",
                use_rerank=True,
                answer_prompt_style="simple",
                multi_snippet_count=1,
            ),
            PipelineVariant(
                key="dual_snippet_direct",
                label="基线 + 双片段证据保留",
                use_rerank=True,
                answer_prompt_style="simple",
                multi_snippet_count=2,
            ),
            PipelineVariant(
                key="dual_snippet_aspect_rerank_direct",
                label="基线 + 双片段证据保留 + 分项重排",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="simple",
                multi_snippet_count=2,
            ),
            PipelineVariant(
                key="dual_snippet_aspect_cover_direct",
                label="基线 + 双片段证据保留 + 分项重排 + 覆盖取证",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="simple",
                multi_snippet_count=2,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
            PipelineVariant(
                key="dual_snippet_aspect_cover_router",
                label="基线 + 双片段证据保留 + 分项重排 + 覆盖取证 + 按题作答",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="task_router",
                multi_snippet_count=2,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
        ),
    },
    "candidate": {
        "output_root": RESULT_ROOT / "候选证据路线_20260517",
        "variants": (
            PipelineVariant(
                key="aspect_cover_router",
                label="当前最佳",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="task_router",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
            PipelineVariant(
                key="aspect_cover_router_labeled",
                label="当前最佳 + 证据标注",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="task_router",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
                rendering_mode="aspect_labeled",
            ),
            PipelineVariant(
                key="aspect_cover_router_window",
                label="当前最佳 + 局部窗口重写",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="task_router",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
                rendering_mode="detail_window",
            ),
        ),
    },
    "structured": {
        "output_root": RESULT_ROOT / "结构化生成路线_20260517",
        "variants": (
            PipelineVariant(
                key="aspect_cover_router",
                label="当前最佳",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                answer_prompt_style="task_router",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
            PipelineVariant(
                key="aspect_cover_clause_guided",
                label="当前最佳 + 模板引导",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                synthesis_mode="clause_guided_aligned",
                answer_prompt_style="aligned",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
            PipelineVariant(
                key="aspect_cover_support_table",
                label="当前最佳 + 支持表综合",
                use_rerank=True,
                rerank_mode="aspect_aware_conservative",
                synthesis_mode="support_table_aligned",
                answer_prompt_style="task_aligned",
                multi_snippet_count=1,
                final_source_count=4,
                complex_source_count=6,
                selection_mode="aspect_cover_v2",
            ),
        ),
    },
}


def run_variant_parallel(
    pipeline: RagExperimentPipeline,
    prepared,
    variant: PipelineVariant,
    *,
    workers: int,
) -> list[Any]:
    if workers <= 1:
        results = []
        total = len(prepared.cases)
        for index, case in enumerate(prepared.cases, start=1):
            if index == 1 or index % 10 == 0 or index == total:
                print(f"[{prepared.name}] {variant.key}: {index}/{total}", flush=True)
            results.append(pipeline.run_case(prepared, case, variant))
        return results

    total = len(prepared.cases)
    ordered_results: list[Any] = [None] * total
    completed = 0
    with ThreadPoolExecutor(max_workers=workers) as executor:
        future_to_index = {
            executor.submit(pipeline.run_case, prepared, case, variant): index
            for index, case in enumerate(prepared.cases)
        }
        for future in as_completed(future_to_index):
            index = future_to_index[future]
            ordered_results[index] = future.result()
            completed += 1
            if completed == 1 or completed % 10 == 0 or completed == total:
                print(f"[{prepared.name}] {variant.key}: {completed}/{total}", flush=True)
    return ordered_results


def load_crud_batch(args: argparse.Namespace):
    cases, docs = load_crud_cases(
        summary_samples=0,
        qa_1doc_samples=0,
        qa_2doc_samples=args.qa_2doc_samples,
        qa_3doc_samples=args.qa_3doc_samples,
        hallu_samples=0,
        negative_samples=0,
        distractor_count=args.distractor_count,
        seed=args.seed,
    )
    expected_count = args.qa_2doc_samples + args.qa_3doc_samples
    if len(cases) != expected_count:
        raise RuntimeError(f"CRUD 子样本当前评测批次数异常，期望 {expected_count}，实际 {len(cases)}")
    return cases, docs


def build_manifest(args: argparse.Namespace, cases, variants: tuple[PipelineVariant, ...], *, suite: str) -> dict[str, Any]:
    return {
        "suite": suite,
        "crud_subset_total": sum(CRUD_SUBSET_SPLITS.values()),
        "crud_subset_splits": CRUD_SUBSET_SPLITS,
        "evaluation_batch_size": len(cases),
        "evaluation_batch_splits": {
            "questanswer_2docs": args.qa_2doc_samples,
            "questanswer_3docs": args.qa_3doc_samples,
        },
        "embedding_model": args.embedding_model,
        "llm_model": args.llm_model,
        "distractor_count": args.distractor_count,
        "seed": args.seed,
        "workers": args.workers,
        "case_ids": [case.case_id for case in cases],
        "variants": [variant.label for variant in variants],
    }


def write_ablation_like_outputs(
    output_root: Path,
    metric_rows: list[dict[str, Any]],
    summaries: list[dict[str, Any]],
    detail_rows: list[dict[str, Any]],
    manifest: dict[str, Any],
) -> None:
    (output_root / "主线替代路线_汇总.json").write_text(
        json.dumps({"summaries": summaries}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (output_root / "主线替代路线_指标表.json").write_text(
        json.dumps(metric_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (output_root / "主线替代路线_逐题明细.json").write_text(
        json.dumps(detail_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    with (output_root / "主线替代路线_指标表.csv").open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "variant",
                "label",
                "sample_count",
                "retrieval_hit_rate_at_1",
                "retrieval_hit_rate_at_3",
                "integration_string_similarity",
                "integration_focus_f1",
                "integration_quality",
                "complex_quality",
                "latency_p50_ms",
                "latency_p95_ms",
            ],
        )
        writer.writeheader()
        writer.writerows(metric_rows)
    (output_root / "评测批次说明.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def _label_to_file_stem(label: str) -> str:
    return (
        label.replace(" + ", "_")
        .replace("+", "_")
        .replace("/", "_")
        .replace("（", "")
        .replace("）", "")
        .replace(" ", "")
    )


def write_rows_outputs(
    output_root: Path,
    summaries: list[dict[str, Any]],
    manifest: dict[str, Any],
) -> None:
    rows = []
    for summary in summaries:
        file_stem = _label_to_file_stem(str(summary["label"]))
        (output_root / f"{file_stem}_汇总.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        rows.append(
            {
                "variant": summary["variant"],
                "label": summary["label"],
                "hit1": summary["retrieval_hit_rate_at_1"],
                "hit3": summary["retrieval_hit_rate_at_3"],
                "integration_string_similarity": summary["integration_string_similarity"],
                "integration_focus_f1": summary["integration_focus_f1"],
                "integration_quality": summary["integration_quality"],
                "complex_quality": summary["complex_quality"],
                "p50": summary["latency_p50_ms"],
            }
        )
    (output_root / "对比总表.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_root / "评测批次说明.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def evaluate_suite(args: argparse.Namespace, suite: str) -> Path:
    suite_config = SUITES[suite]
    output_root = suite_config["output_root"]
    variants: tuple[PipelineVariant, ...] = suite_config["variants"]
    ensure_dir(output_root)
    cache_root = ensure_dir(ROOT / ".cache")

    cases, docs = load_crud_batch(args)
    pipeline = RagExperimentPipeline(
        cache_root=cache_root,
        embedding_model=args.embedding_model,
        llm_model=args.llm_model,
    )
    prepared = pipeline.prepare_dataset(
        f"crud_{suite}",
        cases,
        docs,
        include_contextual=False,
        include_parent_child=False,
        include_query_rewrite=False,
    )

    summaries: list[dict[str, Any]] = []
    detail_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for variant in variants:
        results = run_variant_parallel(pipeline, prepared, variant, workers=max(1, args.workers))
        evaluation = evaluate_crud_results(
            variant.key,
            results,
            cases,
            ragas_case_ids=(),
            qa_ragas_case_ids=(),
            multidoc_ragas_case_ids=(),
            enable_ragas=False,
            semantic_model_name=args.embedding_model,
        )
        summary = dict(evaluation.summary)
        summary["label"] = variant.label
        summaries.append(summary)
        detail_rows.extend(evaluation.detail_rows)
        metric_rows.append(
            {
                "variant": variant.key,
                "label": variant.label,
                "sample_count": summary["sample_count"],
                "retrieval_hit_rate_at_1": summary["retrieval_hit_rate_at_1"],
                "retrieval_hit_rate_at_3": summary["retrieval_hit_rate_at_3"],
                "integration_string_similarity": summary["integration_string_similarity"],
                "integration_focus_f1": summary["integration_focus_f1"],
                "integration_quality": summary["integration_quality"],
                "complex_quality": summary["complex_quality"],
                "latency_p50_ms": summary["latency_p50_ms"],
                "latency_p95_ms": summary["latency_p95_ms"],
            }
        )

    manifest = build_manifest(args, cases, variants, suite=suite)
    if suite == "dual_snippet":
        write_ablation_like_outputs(output_root, metric_rows, summaries, detail_rows, manifest)
    else:
        write_rows_outputs(output_root, summaries, manifest)
    return output_root


def main() -> None:
    parser = argparse.ArgumentParser(description="运行证伪实验中的主线替代路线对比。")
    parser.add_argument("--suite", choices=("dual_snippet", "candidate", "structured", "all"), default="all")
    parser.add_argument("--embedding-model", default=DEFAULT_EMBEDDING_MODEL)
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    parser.add_argument("--qa-2doc-samples", type=int, default=20)
    parser.add_argument("--qa-3doc-samples", type=int, default=20)
    parser.add_argument("--distractor-count", type=int, default=600)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--workers", type=int, default=3)
    args = parser.parse_args()

    suites = tuple(SUITES) if args.suite == "all" else (args.suite,)
    output_roots = [str(evaluate_suite(args, suite)) for suite in suites]
    print(json.dumps({"output_roots": output_roots}, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 双片段路线：评测批次说明

- 文件：`../results/01_主线替代路线/双片段路线_20260517/评测批次说明.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/双片段路线_20260517/评测批次说明.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "dataset": "crud_dual_snippet_batch",
  "crud_subset_total": 1594,
  "crud_subset_splits": {
    "questanswer_2docs": 797,
    "questanswer_3docs": 797
  },
  "evaluation_batch_size": 40,
  "evaluation_batch_splits": {
    "questanswer_2docs": 20,
    "questanswer_3docs": 20
  },
  "embedding_model": "bge-m3:latest",
  "llm_model": "qwen2.5:7b-instruct",
  "distractor_count": 600,
  "seed": 42,
  "workers": 3,
  "case_ids": [
    "questanswer_2docs_001",
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_004",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_007",
    "questanswer_2docs_008",
    "questanswer_2docs_009",
    "questanswer_2docs_010",
    "questanswer_2docs_011",
    "questanswer_2docs_012",
    "questanswer_2docs_013",
    "questanswer_2docs_014",
    "questanswer_2docs_015",
    "questanswer_2docs_016",
    "questanswer_2docs_017",
    "questanswer_2docs_018",
    "questanswer_2docs_019",
    "questanswe

### 双片段路线：汇总

- 文件：`../results/01_主线替代路线/双片段路线_20260517/主线替代路线_汇总.json`

In [2]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/双片段路线_20260517/主线替代路线_汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "summaries": [
    {
      "variant": "baseline_rrf_rerank_direct",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "qa_similarity": 0.8798,
      "qa_string_similarity": 0.4469,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
      "overall_similarity": 0.8798,
      "overall_string_similarity": 0.4469,
      "summary_similarity": 0.0,
      "summary_string_similarity": 0.0,
      "noise_robustness": 0.0,
  

### 双片段路线：指标表

- 文件：`../results/01_主线替代路线/双片段路线_20260517/主线替代路线_指标表.csv`

In [3]:
from pathlib import Path
import csv

path = Path('../results/01_主线替代路线/双片段路线_20260517/主线替代路线_指标表.csv')
with path.open('r', encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print(f'rows={len(rows)} preview={min(len(rows), 8)}')
for row in rows[:8]:
    print(row)


variant,label,sample_count,retrieval_hit_rate_at_1,retrieval_hit_rate_at_3,integration_string_similarity,integration_focus_f1,integration_quality,complex_quality,latency_p50_ms,latency_p95_ms
baseline_rrf_rerank_direct,基线,40,0.875,1.0,0.4469,0.4641,0.6252,0.6252,5635.47,11903.83
dual_snippet_direct,基线 + 双片段证据保留,40,0.9,0.975,0.4043,0.4344,0.595,0.595,6149.67,8853.27
dual_snippet_aspect_rerank_direct,基线 + 双片段证据保留 + 分项重排,40,0.9,0.95,0.4216,0.462,0.6092,0.6092,4245.88,8304.92
dual_snippet_aspect_cover_direct,基线 + 双片段证据保留 + 分项重排 + 覆盖取证,40,0.9,0.975,0.423,0.4202,0.5997,0.5997,7053.98,9368.15
dual_snippet_aspect_cover_router,基线 + 双片段证据保留 + 分项重排 + 覆盖取证 + 按题作答,40,0.9,0.975,0.4909,0.5276,0.6577,0.6577,5516.12,7160.06


### 候选证据路线：评测批次说明

- 文件：`../results/01_主线替代路线/候选证据路线_20260517/评测批次说明.json`

In [4]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/候选证据路线_20260517/评测批次说明.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "dataset": "crud_candidate_routes_batch",
  "crud_subset_total": 1594,
  "crud_subset_splits": {
    "questanswer_2docs": 797,
    "questanswer_3docs": 797
  },
  "evaluation_batch_size": 40,
  "evaluation_batch_splits": {
    "questanswer_2docs": 20,
    "questanswer_3docs": 20
  },
  "embedding_model": "bge-m3:latest",
  "llm_model": "qwen2.5:7b-instruct",
  "distractor_count": 600,
  "seed": 42,
  "workers": 3,
  "case_ids": [
    "questanswer_2docs_001",
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_004",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_007",
    "questanswer_2docs_008",
    "questanswer_2docs_009",
    "questanswer_2docs_010",
    "questanswer_2docs_011",
    "questanswer_2docs_012",
    "questanswer_2docs_013",
    "questanswer_2docs_014",
    "questanswer_2docs_015",
    "questanswer_2docs_016",
    "questanswer_2docs_017",
    "questanswer_2docs_018",
    "questanswer_2docs_019",
    "questa

### 候选证据路线：对比总表

- 文件：`../results/01_主线替代路线/候选证据路线_20260517/对比总表.json`

In [5]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/候选证据路线_20260517/对比总表.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


[
  {
    "variant": "aspect_cover_router",
    "label": "当前最佳",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.5111,
    "integration_focus_f1": 0.6118,
    "integration_quality": 0.6884,
    "complex_quality": 0.6884,
    "p50": 3758.47
  },
  {
    "variant": "aspect_cover_router_labeled",
    "label": "当前最佳 + 证据标注",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.4055,
    "integration_focus_f1": 0.4904,
    "integration_quality": 0.5889,
    "complex_quality": 0.5889,
    "p50": 3694.41
  },
  {
    "variant": "aspect_cover_router_window",
    "label": "当前最佳 + 局部窗口重写",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.4175,
    "integration_focus_f1": 0.4764,
    "integration_quality": 0.6002,
    "complex_quality": 0.6002,
    "p50": 3746.75
  }
]


### 结构化生成路线：评测批次说明

- 文件：`../results/01_主线替代路线/结构化生成路线_20260517/评测批次说明.json`

In [6]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/结构化生成路线_20260517/评测批次说明.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "dataset": "crud_structured_routes_batch",
  "crud_subset_total": 1594,
  "crud_subset_splits": {
    "questanswer_2docs": 797,
    "questanswer_3docs": 797
  },
  "evaluation_batch_size": 40,
  "evaluation_batch_splits": {
    "questanswer_2docs": 20,
    "questanswer_3docs": 20
  },
  "embedding_model": "bge-m3:latest",
  "llm_model": "qwen2.5:7b-instruct",
  "distractor_count": 600,
  "seed": 42,
  "workers": 3,
  "case_ids": [
    "questanswer_2docs_001",
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_004",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_007",
    "questanswer_2docs_008",
    "questanswer_2docs_009",
    "questanswer_2docs_010",
    "questanswer_2docs_011",
    "questanswer_2docs_012",
    "questanswer_2docs_013",
    "questanswer_2docs_014",
    "questanswer_2docs_015",
    "questanswer_2docs_016",
    "questanswer_2docs_017",
    "questanswer_2docs_018",
    "questanswer_2docs_019",
    "quest

### 结构化生成路线：对比总表

- 文件：`../results/01_主线替代路线/结构化生成路线_20260517/对比总表.json`

In [7]:
from pathlib import Path
import json

path = Path('../results/01_主线替代路线/结构化生成路线_20260517/对比总表.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


[
  {
    "variant": "aspect_cover_router",
    "label": "当前最佳",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.5133,
    "integration_focus_f1": 0.567,
    "integration_quality": 0.6707,
    "complex_quality": 0.6707,
    "p50": 3912.88
  },
  {
    "variant": "aspect_cover_clause_guided",
    "label": "当前最佳 + 模板引导",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.4857,
    "integration_focus_f1": 0.4982,
    "integration_quality": 0.6546,
    "complex_quality": 0.6546,
    "p50": 5759.04
  },
  {
    "variant": "aspect_cover_support_table",
    "label": "当前最佳 + 支持表综合",
    "hit1": 0.9,
    "hit3": 0.975,
    "integration_string_similarity": 0.4441,
    "integration_focus_f1": 0.5324,
    "integration_quality": 0.6492,
    "complex_quality": 0.6492,
    "p50": 3207.83
  }
]
